In [ ]:
#install correct packages if needed
#%pip install pandas matplotlib numpy seaborn networkx massql openpyxl pyteomics
#need to be running also in massql_env (Python 3.10.12)

In [ ]:
import numpy as np
from pyteomics import mzml
import importlib
import pandas as pd
import matplotlib.pyplot as plt
import colorsys
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np
import shutil
from datetime import datetime, timedelta
import sys
from matplotlib.colors import ListedColormap
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx
from pathlib import Path
import glob, os, re
from filter_precursor_range_script import filter_precursor_range
from plotting_ind_heatmap import plot_heatmaps 
from summary_builder import explode_matches_to_per_file
from summary_builder import explode_matches_to_per_file
from summary_builder import add_ms1_auc_from_points
from summary_builder import pool_auc_back_to_matches


#this tells python to reload all imported modules automatically before every cell execution 
%load_ext autoreload
%autoreload 2

#Add your project folder to the Python path
sys.path.append("/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project")

import massql_utils as mu
importlib.reload(mu)   # reload if you edit massql_utils.py

import adduct_finder as af
importlib.reload(af)   # reload if you edit adduct_finder.py

import importlib, summary_builder
importlib.reload(summary_builder)

Load in files here. Use hardcoded paths if needed (True) or find in folder (False). 

In [ ]:
#Add files to process
from pathlib import Path

#test 01-07-2025
from pathlib import Path

USE_HARDCODED = True  # flip to False to use mzml_dir + names. If True this will use the hardcoded files you have listed!

HARDCODED_FILES = [
    "/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project/CPCC_632.mzML",
    "/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project/CPCC_300.mzML",
]

mzml_dir = Path("/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/mzml")
names = [
    "ID-06355F1",  #  keep this one for now
    "ID-06279F1","ID-06279F2","ID-06283F1","ID-06283F2",
    "ID-06287F1","ID-06287F2","ID-06291F1","ID-06291F2",
    "ID-06295F1","ID-06295F2","ID-06299F1","ID-06299F2",
    "ID-06303F1","ID-06303F2","ID-06307F1","ID-06307F2",
    "ID-06311F","ID-06315F1","ID-06315F2","ID-06319F1",
    "ID-06319F2","ID-06323F1","ID-06323F2","ID-06327F1",
    "ID-06327F2","ID-06331F1","ID-06331F2","ID-06335F1",
    "ID-06335F2","ID-06339F1","ID-06339F2","ID-06343F1",
    "ID-06343F2","ID-06347F1","ID-06347F2","ID-06351F1",
    "ID-06351F2","ID-06355F2","ID-06359F1","ID-06359F2",
    "ID-06363F1","ID-06363F2","ID-06366F1","ID-06366F2",
    "ID-06370F1","ID-06370F2","ID-06374F1","ID-06374F2",
    "ID-06378F1","ID-06378F2","ID-06382F1","ID-06382F2",
    "ID-06386F1","ID-06386F2",
]

if USE_HARDCODED:
    FILES = HARDCODED_FILES
else:
    FILES = []
    for name in names:
        hits = sorted(mzml_dir.glob(f"*{name}.mzML"))
        if len(hits) == 0:
            print(f"WARNING: no file found for {name}")
        FILES.extend(str(p) for p in hits)

print("Loaded", len(FILES), "mzML files")


#Ions to check (individual)
MCIONS = [135.0804, 163.1113, 213.0870] 
MPIONS = [184.06, 215.1192, 243.1127, 134.0961, 181.1331, 169.0967, 150.0912, 167.1178, 454.15]
ARIONS = [112.0964, 140.1066, 334.0838, 300.1232, 284.1268, 250.1440, 221.1646, 281.1914, 314.2199]
ABIONS = [114.0550, 164.1069, 192.1019, 233.1285, 263.1391, 362.2075]
MVIONS = [136.0759, 136.0729, 159.0908, 159.0935]
AGIONS = [120.0807, 144.0109, 188.1428, 213.0685, 199.0503, 267.1483]
MGIONS = [100.1122, 134.0727, 168.0341, 201.9952, 114.1278, 148.0888, 182.0494, 128.1423, 162.1039, 196.0639, 142.1590, 176.1195, 210.0795]

TOL = 0.01
POLARITY = "POSITIVE"   # or "NEGATIVE" or None
RT_WINDOW = (2.0, 25)   # optional retention time window (min)


ION_TO_LABEL = {
    184.06:   "MP NMeTyrCl 184.06",
    215.1192: "MP AhpPhe-CO-H2O 215.12",
    243.1127: "MP AhpPhe-H2O 243.11",
    134.0961: "MP NMePhe 134.10",
    181.1331: "MP AhpLxx-CO-H2O 181.13",
    169.0967: "MP AhpThr-CO-H2O 169.10",
    150.0912: "MP NMeTyr 150.09",
    167.1178: "MP AhpVal-CO-H2O 167.12",
    454.15:   "MP Phe-Ahp-complete 454.15",
    135.0804: "MC adda fragment 135.0804",
    163.1113: "MC adda fragment 163.1113",
    213.0870: "MC Mdha 213.0870",
    112.0964: "AR Choi-h2o 112.0964",
    140.1066: "AR Choi 140.1066",
    334.0838: "AR Cl-Hpla-Tyr-CO 334.0838",
    300.1232: "AR Hpla-Tyr-CO 300.1232",
    284.1268: "AR Hlpa-Phe-CO 284.1268",
    250.1440: "AR Hlpa-Leu-CO 250.1440",
    221.1646: "AR Hlpa-Leu 221.1646",
    281.1914: "AR Agma 281.1914",
    314.2199: "AR OH-Choi-Agma 314.2199",
    114.0550: "AB CO + Arg- CH5H3 -CO 114.0550",
    164.1069: "AB NMe-HTyr 164.1069",
    192.1019: "AB CO-NMe-Hty 192.1019",
    233.1285: "AB Phe + MeAla +H 233.1285",
    263.1391: "AB HTyr +MeAla +H 263.1391",
    362.2075: "AB HTyr+ MeAla +Val +H 362.2075",
    136.0759: "MV Tyr 136.0759",
    136.0729: "MV Tyr 136.0729",
    159.0908: "MV Try 159.0908",
    159.0935: "MV Tyr 159.0935",
    120.0807: "AG Phe 120.0807",
    144.0109: "AG Tzc 144.0109",
    188.1428: "AG 188.1428",
    213.0685: "AG Pro-Tzc 213.0685",
    199.0503: "AG Tzl-OH 199.0503",
    267.1483: "AG 267.1483",
    100.1122: "MG Ahoa 100.1122",
    134.0727: "MG Ahoa (Cl) 134.0727",
    168.0341: "MG Ahoa (Di Cl) 168.0341",
    201.9955: "MG Ahoa (Tri Cl) 201.9955",
    114.1278: "MG NMe Ahoa 114.1278",
    148.0888: "MG NMe Ahoa (Cl) 148.0888", 
    182.0494: "MG NMe Ahoa (Di Cl) 182.0494",
    126.1423: "MG Ahda 128.1423",
    162.1039: "MG Ahda (Cl) 162.1039",
    196.0639: "MG Ahda (Di Cl) 196.0639",
    142.1590: "MG NMe Ahda 142.1590",
    176.1195: "MG NMe Ahda (Cl) 176.1196",
    210.0795: "MG NMe Ahda (Di Cl) 201.0795",
}


#Ion combinations (all ions must co-occur in the same MS2 spectrum)
MPCOMBOS = [
    (184.06, 215.1192),
    (184.06, 215.1192, 243.1127),
    (181.1331, 134.0961),
]

#Load files (per-file + merged) to read in raw data
data = mu.load_files(FILES)

# Per-file access
per_file = data["per_file"]            
ms1_all, ms2_all = data["merged"]

Extracts out MS1 EIC. Can alter RT and intensity thresholds here. 

In [ ]:
# --- MS1 extraction from mzML -> point table CSV (with optional RT filter) ---
# Assumes you already have:
#   FILES = [...]  # list[str] of mzML paths
#
# Output CSV columns:
#   source_file, scan, rt, precmz, i, mslevel, polarity
#
# Install (if needed):
# %pip install pyteomics numpy pandas lxml



def _parse_scan_number(spec: dict) -> int | None:
    """Try to extract integer scan number from mzML 'id' string."""
    sid = spec.get("id", "")
    if isinstance(sid, str) and "scan=" in sid:
        try:
            return int(sid.split("scan=")[-1].split()[0])
        except Exception:
            return None
    return None


def _get_rt_minutes(spec: dict, assume_time_unit: str = "min") -> float | None:
    """
    Get RT as minutes.
    pyteomics often provides 'scan start time' already.
    """
    rt = spec.get("scan start time", None)
    if rt is None:
        try:
            sc = spec.get("scanList", {}).get("scan", [])
            if isinstance(sc, dict):
                sc = [sc]
            if sc:
                rt = sc[0].get("scan start time", None)
        except Exception:
            rt = None

    if rt is None:
        return None

    rt = float(rt)
    if assume_time_unit.lower().startswith("sec"):
        rt = rt / 60.0
    return rt


def _guess_polarity(spec: dict) -> int | None:
    """
    Heuristic polarity detection.
    Returns 1 (positive), -1 (negative), or None.
    """
    txt = str(spec).lower()
    if "positive scan" in txt:
        return 1
    if "negative scan" in txt:
        return -1
    return None


def extract_ms1_points_from_mzml(
    mzml_path: str | Path,
    *,
    mz_round: int | None = None,        # e.g., 6 for smaller CSVs
    intensity_min: float = 0.0,         # e.g., 1000 to drop low-intensity noise
    assume_time_unit: str = "min",      # "min" or "sec"
    rt_min: float | None = None,        # minutes
    rt_max: float | None = None,        # minutes
) -> pd.DataFrame:
    """
    Extract MS1 points from an mzML file into a long table:
      source_file, scan, rt, precmz, i, mslevel, polarity

    RT filtering (rt_min/rt_max) is applied PER SCAN (fast), before expanding points.
    """
    mzml_path = str(mzml_path)
    src = Path(mzml_path).name

    out_source = []
    out_scan   = []
    out_rt     = []
    out_mz     = []
    out_i      = []
    out_ms1    = []
    out_pol    = []

    with mzml.MzML(mzml_path) as reader:
        for spec in reader:
            mslevel = spec.get("ms level") or spec.get("msLevel") or spec.get("mslevel")
            if mslevel is None or int(mslevel) != 1:
                continue

            rt = _get_rt_minutes(spec, assume_time_unit=assume_time_unit)
            if rt is None:
                continue

            # ---- RT filter (minutes) ----
            if (rt_min is not None) and (rt < rt_min):
                continue
            if (rt_max is not None) and (rt > rt_max):
                continue

            scan_num = _parse_scan_number(spec)
            pol = _guess_polarity(spec)

            mzs = spec.get("m/z array", None)
            ints = spec.get("intensity array", None)
            if mzs is None or ints is None:
                continue

            mzs = np.asarray(mzs, dtype=float)
            ints = np.asarray(ints, dtype=float)

            if intensity_min > 0:
                keep = ints >= float(intensity_min)
                if not np.any(keep):
                    continue
                mzs = mzs[keep]
                ints = ints[keep]

            if mz_round is not None:
                mzs = np.round(mzs, int(mz_round))

            n = mzs.size
            if n == 0:
                continue

            # vectorized-ish appends (faster than per-point dict rows)
            out_source.extend([src] * n)
            out_scan.extend([scan_num] * n)
            out_rt.extend([rt] * n)
            out_mz.extend(mzs.tolist())
            out_i.extend(ints.tolist())
            out_ms1.extend([1] * n)
            out_pol.extend([pol] * n)

    return pd.DataFrame(
        {
            "source_file": out_source,
            "scan": out_scan,
            "rt": out_rt,
            "precmz": out_mz,   # keeping your naming; this is MS1 m/z
            "i": out_i,
            "mslevel": out_ms1,
            "polarity": out_pol,
        }
    )


# -----------------------------
# Run across your FILES and write CSV
# -----------------------------
OUT_DIR = Path(".")  # change if you want: Path("/some/output/folder")
ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUT_CSV = OUT_DIR / f"ms1_points_{ts}.csv"

# OPTIONAL: set RT window here (minutes). Set to None to disable.
RT_MIN = None   # ex 2.0
RT_MAX = None   # ex 11.0

# OPTIONAL: size/noise controls
MZ_ROUND = None         # e.g., 6
INTENSITY_MIN = 0.0     # e.g., 1000.0
ASSUME_TIME_UNIT = "min"

all_dfs = []
for p in FILES:
    p = str(p)
    print("[INFO] extracting MS1:", Path(p).name)
    df = extract_ms1_points_from_mzml(
        p,
        mz_round=MZ_ROUND,
        intensity_min=INTENSITY_MIN,
        assume_time_unit=ASSUME_TIME_UNIT,
        rt_min=RT_MIN,
        rt_max=RT_MAX,
    )
    print("   rows:", len(df))
    all_dfs.append(df)

ms1_points = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
print("[INFO] total MS1 rows:", len(ms1_points))

ms1_points.to_csv(OUT_CSV, index=False)
print("[DONE] wrote:", OUT_CSV)

Anabaenopeptin Search: 
alter this for reference compounds: 
5)Reference compound MS1 AUC per file (FROM MS1 POINTS)
#edit this for own refernece compound!!!!!
REF_MZ = 192.20
REF_TOL = 0.1 

In [ ]:
# ============================================================
# AB pipeline (MassQL diagnostic ions) + MS1 AUC from MS1 points
#updated for 1/5/2025 (12pm)
# ============================================================
# Assumes you already defined:
#   FILES, mu, ABIONS, TOL, POLARITY, RT_WINDOW, ION_TO_LABEL
#
# Also assumes you've already generated MS1 point tables earlier:
#   ms1_points_*.csv with columns:
#   source_file, scan, rt, precmz, i, mslevel, polarity
#
# Outputs:
#   individual_hits_AB_<ts>.csv
#   reference_ms1_auc_AB_<ts>.csv
#   AUC_outputs/AB_indiv_summary_perfile_with_ms1_auc_<ts>.csv
#   AUC_outputs/AB_indiv_summary_pooled_ms1_auc_<ts>.csv
#   (plus your plots)

import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from summary_builder import (
    make_summary_ind,
    explode_matches_to_per_file,
    add_ms1_auc_from_points,
    pool_auc_back_to_matches,
)

# ------------------------------------------------------------
# 0) Timestamp for all outputs in this run
# ------------------------------------------------------------
ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# ------------------------------------------------------------
# 1) MassQL query + run across files (AB diagnostic ions)
# ------------------------------------------------------------
query = mu.build_massql_query(
    ions_mz=ABIONS,
    tol_mz=TOL,
    polarity=POLARITY,
    rt_window=None
)

ind_hits = mu.run_across_files_individual(
    FILES, ABIONS, tol_mz=TOL, polarity=POLARITY, rt_window=RT_WINDOW
)

ind_hits_l = mu.add_AB_labels(ind_hits, ION_TO_LABEL, label_col="CyanopeptideClass_AB")

ind_csv = f"individual_hits_AB_{ts}.csv"
ind_hits_l.to_csv(ind_csv, index=False)
print("[DONE] wrote:", ind_csv)

# ------------------------------------------------------------
# 2) Plots
# ------------------------------------------------------------
from rt_histograms import plot_rt_histograms
from rt_histograms import load_latest_hits

# Reload latest (your helper likely picks up the CSV you just wrote)
ind_hits_l = load_latest_hits()

plot_rt_histograms(ind_hits_l, ION_TO_LABEL, out_dir_root="RT_histogram_plots")

from cyanopeptide_counts_plots import plot_indiv_counts
plot_indiv_counts(ind_hits_l, out_dir="plots_diagnostic_ion_counts")

from rt_mz_plot import plot_precursor_rt, plot_per_file_legend
fig, ax = plot_precursor_rt(ind_hits_l, ion_to_label=ION_TO_LABEL, save=True)
plot_per_file_legend(ind_hits_l, save=True)

from plotting_ind_heatmap import plot_heatmaps
plot_heatmaps(
    ind_hits_l,
    ion_to_label=ION_TO_LABEL,
    save=True,
    out_dir="Heatmap_Individual_ion_plots",
    fmt="png"
)

from filter_precursor_range_script import filter_precursor_range

ind_hits_filtered_range = filter_precursor_range(ind_hits_l, min_mz=650, max_mz=1400)
plot_heatmaps(
    ind_hits_filtered_range,
    ion_to_label=ION_TO_LABEL,
    BIN_WIDTH=0.1,
    merged=True,
    per_file=True
)

# ------------------------------------------------------------
# 3) Build merged precursor summary from MS2 hits
# ------------------------------------------------------------
indiv_summary = make_summary_ind(ind_hits_l, ion_to_label=ION_TO_LABEL)

# ------------------------------------------------------------
# 4) Load the most recent MS1 points CSV automatically
# ------------------------------------------------------------
ms1_csv = max(Path(".").glob("ms1_points_*.csv"), key=lambda p: p.stat().st_mtime)
print("[INFO] Using MS1 points file:", ms1_csv)

ms1_points = pd.read_csv(ms1_csv)

# normalize filenames to basenames (critical to match summary_builder logic)
ms1_points["source_file"] = ms1_points["source_file"].astype(str).apply(lambda x: Path(x).name)

# sanity checks
print("[INFO] MS1 points rows:", len(ms1_points))
print("[INFO] MS1 points unique files:", ms1_points["source_file"].nunique())
if "mslevel" in ms1_points.columns:
    print("[INFO] MS levels:", ms1_points["mslevel"].value_counts().head())

# ------------------------------------------------------------
# 5) Reference compound MS1 AUC per file (FROM MS1 POINTS)
# ------------------------------------------------------------
#edit this for own refernece compound!!!!!
REF_MZ = 192.20
REF_TOL = 0.1
REF_RT_MIN, REF_RT_MAX = RT_WINDOW  # minutes

# Optional: polarity filter on reference extraction
# (Only if your MS1 table has polarity populated reliably)
POL_FOR_REF = POLARITY if isinstance(POLARITY, int) else None

ref_auc = {}
for fname, sub in ms1_points.groupby("source_file", sort=False):
    win = sub[
        (sub["mslevel"] == 1)
        & (sub["precmz"].between(REF_MZ - REF_TOL, REF_MZ + REF_TOL))
        & (sub["rt"].between(REF_RT_MIN, REF_RT_MAX))
    ]
    if POL_FOR_REF is not None and "polarity" in sub.columns:
        win = win[win["polarity"] == POL_FOR_REF]

    if win.empty:
        ref_auc[fname] = 0.0
        continue

    trace = (
        win.groupby("rt", as_index=False)["i"]
           .sum()
           .sort_values("rt")
    )

    ref_auc[fname] = 0.0 if len(trace) < 2 else float(np.trapz(trace["i"], trace["rt"]))

missing_refs = [k for k, v in ref_auc.items() if v == 0.0]
print("\n[INFO] Files missing reference compound (MS1 AUC==0):")
for f in missing_refs:
    print(" -", f)

ref_auc_df = pd.DataFrame([{"source_file": k, "ref_ms1_auc": v} for k, v in ref_auc.items()])
ref_auc_csv = f"reference_ms1_auc_AR_{ts}.csv"
ref_auc_df.to_csv(ref_auc_csv, index=False)
print("[DONE] wrote:", ref_auc_csv)

# ------------------------------------------------------------
# 6) Compute MS1 AUC for each (feature,file) from MS1 points
# ------------------------------------------------------------
perfile_summary = explode_matches_to_per_file(indiv_summary)

perfile_summary_auc = add_ms1_auc_from_points(
    perfile_summary,
    ms1_points,
    tol_mz=TOL,          # same tolerance you use elsewhere
    rt_pad=0.25,
    polarity=POLARITY if isinstance(POLARITY, int) else None,  # 1/-1/None
    intensity_col="i",   # integrate raw intensity
    mz_col="precmz"
)

# Add reference + normalized column
perfile_summary_auc = perfile_summary_auc.merge(ref_auc_df, on="source_file", how="left")
perfile_summary_auc["ms1_auc_over_ref"] = np.where(
    (perfile_summary_auc["ref_ms1_auc"].notna()) & (perfile_summary_auc["ref_ms1_auc"] > 0),
    perfile_summary_auc["ms1_auc"] / perfile_summary_auc["ref_ms1_auc"],
    np.nan
)

# Optional: pool back to one row per merged precursor (sum across files)
pooled_auc = pool_auc_back_to_matches(perfile_summary_auc)

# ------------------------------------------------------------
# 7) Save AUC outputs
# ------------------------------------------------------------
out_dir = Path("AUC_outputs")
out_dir.mkdir(exist_ok=True)

perfile_out = out_dir / f"AB_indiv_summary_perfile_with_ms1_auc_{ts}.csv"
pooled_out  = out_dir / f"AB_indiv_summary_pooled_ms1_auc_{ts}.csv"

perfile_summary_auc.to_csv(perfile_out, index=False)
pooled_auc.to_csv(pooled_out, index=False)

print("[DONE] Saved per-file AUC table:", perfile_out)
print("[DONE] Saved pooled AUC table:  ", pooled_out)

# ------------------------------------------------------------
# 8) Downstream scatter plot — use indiv_summary
# ------------------------------------------------------------
from indiv_combo_dot_plot import plot_indiv_scatter
fig, ax, path = plot_indiv_scatter(indiv_summary, out_dir="DotPlot_individual_ion_plots", fmt="png")

# ------------------------------------------------------------
# 9) (Optional) Your adduct pipeline block (unchanged)
# ------------------------------------------------------------
import adduct_pipeline as ap
import adduct_finder as af

def make_summary_ind_with_labels(df, *args, **kwargs):
    return make_summary_ind(df, *args, ion_to_label=ION_TO_LABEL, **kwargs)

filtered_ind = ind_hits_l[(ind_hits_l["precmz"] >= 650) & (ind_hits_l["precmz"] <= 1200)]

merged_summary, merged_edges, G = ap.run_merged(
    filtered_ind,
    make_summary_ind=make_summary_ind_with_labels,
    af_module=af,
    mz_col="merged_precmz",
    charge_col="charge",
    rt_col="rt_median",
    out_dir="Adduct_and_summary_outputs",
    save_graph=True,
)

# ------------------------------------------------------------
# 10) Library matching + tiles (unchanged)
# ------------------------------------------------------------
from cyanometdb_match import (
    load_library, read_any_table, match_ms1_to_lib, select_ms1_columns,
    _latest_indiv_summary, write_outputs, plot_matched_tiles
)

LIB_XLSX = "/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project/CyanoMetDB_Version03.xlsx"

lib_df  = load_library(LIB_XLSX, class_filter="Anabaenopeptin", sheet_index=1)
ms1_csv_for_match = _latest_indiv_summary()
ms1_df_for_match  = read_any_table(ms1_csv_for_match)

ms1_sel = select_ms1_columns(ms1_df_for_match)

matches = match_ms1_to_lib(ms1_sel, lib_df, tol_da=0.1)
matched_only = matches[matches["Compound identifier"].notna()].copy()

ts2 = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir_ts, paths = write_outputs(matches, matched_only, out_dir="CyanoMetDB_matches_out", ts=ts2)
plot_matched_tiles(matched_only, out_dir_ts=out_dir_ts, ts=ts2)

# ------------------------------------------------------------
# 11) Your remaining utilities (unchanged)
# ------------------------------------------------------------
from sum_intensity_from_scans import sum_intensities
df, outfile = sum_intensities()

from ms2_tilemap_intensities import plot_has_tilemap_from_latest
df, run_dir, summary_file = plot_has_tilemap_from_latest(ion_to_label=ION_TO_LABEL)

from group_run_files import group_files_by_timestamp
group_files_by_timestamp(custom_name="AB", minutes=7)


Aeruginosin search

In [ ]:
# ============================================================
# AR pipeline (MassQL diagnostic ions) + MS1 AUC from MS1 points
#updated for 1/5/2025 (12pm)
# ============================================================
# Assumes you already defined:
#   FILES, mu, ARIONS, TOL, POLARITY, RT_WINDOW, ION_TO_LABEL
#
# Also assumes you've already generated MS1 point tables earlier:
#   ms1_points_*.csv with columns:
#   source_file, scan, rt, precmz, i, mslevel, polarity
#
# Outputs:
#   individual_hits_AR_<ts>.csv
#   reference_ms1_auc_AR_<ts>.csv
#   AUC_outputs/AR_indiv_summary_perfile_with_ms1_auc_<ts>.csv
#   AUC_outputs/AR_indiv_summary_pooled_ms1_auc_<ts>.csv
#   (plus your plots)

import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from summary_builder import (
    make_summary_ind,
    explode_matches_to_per_file,
    add_ms1_auc_from_points,
    pool_auc_back_to_matches,
)

# ------------------------------------------------------------
# 0) Timestamp for all outputs in this run
# ------------------------------------------------------------
ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# ------------------------------------------------------------
# 1) MassQL query + run across files (AR diagnostic ions)
# ------------------------------------------------------------
query = mu.build_massql_query(
    ions_mz=ARIONS,
    tol_mz=TOL,
    polarity=POLARITY,
    rt_window=None
)

ind_hits = mu.run_across_files_individual(
    FILES, ARIONS, tol_mz=TOL, polarity=POLARITY, rt_window=RT_WINDOW
)

ind_hits_l = mu.add_AR_labels(ind_hits, ION_TO_LABEL, label_col="CyanopeptideClass_AR")

ind_csv = f"individual_hits_AR_{ts}.csv"
ind_hits_l.to_csv(ind_csv, index=False)
print("[DONE] wrote:", ind_csv)

# ------------------------------------------------------------
# 2) Plots
# ------------------------------------------------------------
from rt_histograms import plot_rt_histograms
from rt_histograms import load_latest_hits

# Reload latest (your helper likely picks up the CSV you just wrote)
ind_hits_l = load_latest_hits()

plot_rt_histograms(ind_hits_l, ION_TO_LABEL, out_dir_root="RT_histogram_plots")

from cyanopeptide_counts_plots import plot_indiv_counts
plot_indiv_counts(ind_hits_l, out_dir="plots_diagnostic_ion_counts")

from rt_mz_plot import plot_precursor_rt, plot_per_file_legend
fig, ax = plot_precursor_rt(ind_hits_l, ion_to_label=ION_TO_LABEL, save=True)
plot_per_file_legend(ind_hits_l, save=True)

from plotting_ind_heatmap import plot_heatmaps
plot_heatmaps(
    ind_hits_l,
    ion_to_label=ION_TO_LABEL,
    save=True,
    out_dir="Heatmap_Individual_ion_plots",
    fmt="png"
)

from filter_precursor_range_script import filter_precursor_range

ind_hits_filtered_range = filter_precursor_range(ind_hits_l, min_mz=650, max_mz=1400)
plot_heatmaps(
    ind_hits_filtered_range,
    ion_to_label=ION_TO_LABEL,
    BIN_WIDTH=0.1,
    merged=True,
    per_file=True
)

# ------------------------------------------------------------
# 3) Build merged precursor summary from MS2 hits
# ------------------------------------------------------------
indiv_summary = make_summary_ind(ind_hits_l, ion_to_label=ION_TO_LABEL)

# ------------------------------------------------------------
# 4) Load the most recent MS1 points CSV automatically
# ------------------------------------------------------------
ms1_csv = max(Path(".").glob("ms1_points_*.csv"), key=lambda p: p.stat().st_mtime)
print("[INFO] Using MS1 points file:", ms1_csv)

ms1_points = pd.read_csv(ms1_csv)

# normalize filenames to basenames (critical to match summary_builder logic)
ms1_points["source_file"] = ms1_points["source_file"].astype(str).apply(lambda x: Path(x).name)

# sanity checks
print("[INFO] MS1 points rows:", len(ms1_points))
print("[INFO] MS1 points unique files:", ms1_points["source_file"].nunique())
if "mslevel" in ms1_points.columns:
    print("[INFO] MS levels:", ms1_points["mslevel"].value_counts().head())

# ------------------------------------------------------------
# 5) Reference compound MS1 AUC per file (FROM MS1 POINTS)
# ------------------------------------------------------------
#edit this for reference compounds!!
REF_MZ = 192.20
REF_TOL = 0.1
REF_RT_MIN, REF_RT_MAX = RT_WINDOW  # minutes

# Optional: polarity filter on reference extraction
# (Only if your MS1 table has polarity populated reliably)
POL_FOR_REF = POLARITY if isinstance(POLARITY, int) else None

ref_auc = {}
for fname, sub in ms1_points.groupby("source_file", sort=False):
    win = sub[
        (sub["mslevel"] == 1)
        & (sub["precmz"].between(REF_MZ - REF_TOL, REF_MZ + REF_TOL))
        & (sub["rt"].between(REF_RT_MIN, REF_RT_MAX))
    ]
    if POL_FOR_REF is not None and "polarity" in sub.columns:
        win = win[win["polarity"] == POL_FOR_REF]

    if win.empty:
        ref_auc[fname] = 0.0
        continue

    trace = (
        win.groupby("rt", as_index=False)["i"]
           .sum()
           .sort_values("rt")
    )

    ref_auc[fname] = 0.0 if len(trace) < 2 else float(np.trapz(trace["i"], trace["rt"]))

missing_refs = [k for k, v in ref_auc.items() if v == 0.0]
print("\n[INFO] Files missing reference compound (MS1 AUC==0):")
for f in missing_refs:
    print(" -", f)

ref_auc_df = pd.DataFrame([{"source_file": k, "ref_ms1_auc": v} for k, v in ref_auc.items()])
ref_auc_csv = f"reference_ms1_auc_AR_{ts}.csv"
ref_auc_df.to_csv(ref_auc_csv, index=False)
print("[DONE] wrote:", ref_auc_csv)

# ------------------------------------------------------------
# 6) Compute MS1 AUC for each (feature,file) from MS1 points
# ------------------------------------------------------------
perfile_summary = explode_matches_to_per_file(indiv_summary)

perfile_summary_auc = add_ms1_auc_from_points(
    perfile_summary,
    ms1_points,
    tol_mz=TOL,          # same tolerance you use elsewhere
    rt_pad=0.25,
    polarity=POLARITY if isinstance(POLARITY, int) else None,  # 1/-1/None
    intensity_col="i",   # integrate raw intensity
    mz_col="precmz"
)

# Add reference + normalized column
perfile_summary_auc = perfile_summary_auc.merge(ref_auc_df, on="source_file", how="left")
perfile_summary_auc["ms1_auc_over_ref"] = np.where(
    (perfile_summary_auc["ref_ms1_auc"].notna()) & (perfile_summary_auc["ref_ms1_auc"] > 0),
    perfile_summary_auc["ms1_auc"] / perfile_summary_auc["ref_ms1_auc"],
    np.nan
)

# Optional: pool back to one row per merged precursor (sum across files)
pooled_auc = pool_auc_back_to_matches(perfile_summary_auc)

# ------------------------------------------------------------
# 7) Save AUC outputs
# ------------------------------------------------------------
out_dir = Path("AUC_outputs")
out_dir.mkdir(exist_ok=True)

perfile_out = out_dir / f"AR_indiv_summary_perfile_with_ms1_auc_{ts}.csv"
pooled_out  = out_dir / f"AR_indiv_summary_pooled_ms1_auc_{ts}.csv"

perfile_summary_auc.to_csv(perfile_out, index=False)
pooled_auc.to_csv(pooled_out, index=False)

print("[DONE] Saved per-file AUC table:", perfile_out)
print("[DONE] Saved pooled AUC table:  ", pooled_out)

# ------------------------------------------------------------
# 8) Downstream scatter plot — use indiv_summary
# ------------------------------------------------------------
from indiv_combo_dot_plot import plot_indiv_scatter
fig, ax, path = plot_indiv_scatter(indiv_summary, out_dir="DotPlot_individual_ion_plots", fmt="png")

# ------------------------------------------------------------
# 9) (Optional) Your adduct pipeline block (unchanged)
# ------------------------------------------------------------
import adduct_pipeline as ap
import adduct_finder as af

def make_summary_ind_with_labels(df, *args, **kwargs):
    return make_summary_ind(df, *args, ion_to_label=ION_TO_LABEL, **kwargs)

filtered_ind = ind_hits_l[(ind_hits_l["precmz"] >= 650) & (ind_hits_l["precmz"] <= 1200)]

merged_summary, merged_edges, G = ap.run_merged(
    filtered_ind,
    make_summary_ind=make_summary_ind_with_labels,
    af_module=af,
    mz_col="merged_precmz",
    charge_col="charge",
    rt_col="rt_median",
    out_dir="Adduct_and_summary_outputs",
    save_graph=True,
)

# ------------------------------------------------------------
# 10) Library matching + tiles (unchanged)
# ------------------------------------------------------------
from cyanometdb_match import (
    load_library, read_any_table, match_ms1_to_lib, select_ms1_columns,
    _latest_indiv_summary, write_outputs, plot_matched_tiles
)

LIB_XLSX = "/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project/CyanoMetDB_Version03.xlsx"

lib_df  = load_library(LIB_XLSX, class_filter="Aeruginosin", sheet_index=1)
ms1_csv_for_match = _latest_indiv_summary()
ms1_df_for_match  = read_any_table(ms1_csv_for_match)

ms1_sel = select_ms1_columns(ms1_df_for_match)

matches = match_ms1_to_lib(ms1_sel, lib_df, tol_da=0.1)
matched_only = matches[matches["Compound identifier"].notna()].copy()

ts2 = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir_ts, paths = write_outputs(matches, matched_only, out_dir="CyanoMetDB_matches_out", ts=ts2)
plot_matched_tiles(matched_only, out_dir_ts=out_dir_ts, ts=ts2)

# ------------------------------------------------------------
# 11) Your remaining utilities (unchanged)
# ------------------------------------------------------------
from sum_intensity_from_scans import sum_intensities
df, outfile = sum_intensities()

from ms2_tilemap_intensities import plot_has_tilemap_from_latest
df, run_dir, summary_file = plot_has_tilemap_from_latest(ion_to_label=ION_TO_LABEL)

from group_run_files import group_files_by_timestamp
group_files_by_timestamp(custom_name="AR", minutes=7)


Microcystin search

In [ ]:
# ============================================================
# MC pipeline (MassQL diagnostic ions) + MS1 AUC from MS1 points
#updated for 1/5/2025 (12pm)
# ============================================================
# Assumes you already defined:
#   FILES, mu, MCIONS, TOL, POLARITY, RT_WINDOW, ION_TO_LABEL
#
# Also assumes you've already generated MS1 point tables earlier:
#   ms1_points_*.csv with columns:
#   source_file, scan, rt, precmz, i, mslevel, polarity
#
# Outputs:
#   individual_hits_MC_<ts>.csv
#   reference_ms1_auc_MC_<ts>.csv
#   AUC_outputs/MC_indiv_summary_perfile_with_ms1_auc_<ts>.csv
#   AUC_outputs/MC_indiv_summary_pooled_ms1_auc_<ts>.csv
#   (plus your plots)

import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from summary_builder import (
    make_summary_ind,
    explode_matches_to_per_file,
    add_ms1_auc_from_points,
    pool_auc_back_to_matches,
)

# ------------------------------------------------------------
# 0) Timestamp for all outputs in this run
# ------------------------------------------------------------
ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# ------------------------------------------------------------
# 1) MassQL query + run across files (MC diagnostic ions)
# ------------------------------------------------------------
query = mu.build_massql_query(
    ions_mz=MCIONS,
    tol_mz=TOL,
    polarity=POLARITY,
    rt_window=None
)

ind_hits = mu.run_across_files_individual(
    FILES, MCIONS, tol_mz=TOL, polarity=POLARITY, rt_window=RT_WINDOW
)

ind_hits_l = mu.add_MC_labels(ind_hits, ION_TO_LABEL, label_col="CyanopeptideClass_MC")

ind_csv = f"individual_hits_MC_{ts}.csv"
ind_hits_l.to_csv(ind_csv, index=False)
print("[DONE] wrote:", ind_csv)

# ------------------------------------------------------------
# 2) Plots
# ------------------------------------------------------------
from rt_histograms import plot_rt_histograms
from rt_histograms import load_latest_hits

# Reload latest (your helper likely picks up the CSV you just wrote)
ind_hits_l = load_latest_hits()

plot_rt_histograms(ind_hits_l, ION_TO_LABEL, out_dir_root="RT_histogram_plots")

from cyanopeptide_counts_plots import plot_indiv_counts
plot_indiv_counts(ind_hits_l, out_dir="plots_diagnostic_ion_counts")

from rt_mz_plot import plot_precursor_rt, plot_per_file_legend
fig, ax = plot_precursor_rt(ind_hits_l, ion_to_label=ION_TO_LABEL, save=True)
plot_per_file_legend(ind_hits_l, save=True)

from plotting_ind_heatmap import plot_heatmaps
plot_heatmaps(
    ind_hits_l,
    ion_to_label=ION_TO_LABEL,
    save=True,
    out_dir="Heatmap_Individual_ion_plots",
    fmt="png"
)

from filter_precursor_range_script import filter_precursor_range

ind_hits_filtered_range = filter_precursor_range(ind_hits_l, min_mz=650, max_mz=1400)
plot_heatmaps(
    ind_hits_filtered_range,
    ion_to_label=ION_TO_LABEL,
    BIN_WIDTH=0.1,
    merged=True,
    per_file=True
)

# ------------------------------------------------------------
# 3) Build merged precursor summary from MS2 hits
# ------------------------------------------------------------
indiv_summary = make_summary_ind(ind_hits_l, ion_to_label=ION_TO_LABEL)

# ------------------------------------------------------------
# 4) Load the most recent MS1 points CSV automatically
# ------------------------------------------------------------
ms1_csv = max(Path(".").glob("ms1_points_*.csv"), key=lambda p: p.stat().st_mtime)
print("[INFO] Using MS1 points file:", ms1_csv)

ms1_points = pd.read_csv(ms1_csv)

# normalize filenames to basenames (critical to match summary_builder logic)
ms1_points["source_file"] = ms1_points["source_file"].astype(str).apply(lambda x: Path(x).name)

# sanity checks
print("[INFO] MS1 points rows:", len(ms1_points))
print("[INFO] MS1 points unique files:", ms1_points["source_file"].nunique())
if "mslevel" in ms1_points.columns:
    print("[INFO] MS levels:", ms1_points["mslevel"].value_counts().head())

# ------------------------------------------------------------
# 5) Reference compound MS1 AUC per file (FROM MS1 POINTS)
# ------------------------------------------------------------
#edit this for reference compounds!!

REF_MZ = 192.20
REF_TOL = 0.1
REF_RT_MIN, REF_RT_MAX = RT_WINDOW  # minutes

# Optional: polarity filter on reference extraction
# (Only if your MS1 table has polarity populated reliably)
POL_FOR_REF = POLARITY if isinstance(POLARITY, int) else None

ref_auc = {}
for fname, sub in ms1_points.groupby("source_file", sort=False):
    win = sub[
        (sub["mslevel"] == 1)
        & (sub["precmz"].between(REF_MZ - REF_TOL, REF_MZ + REF_TOL))
        & (sub["rt"].between(REF_RT_MIN, REF_RT_MAX))
    ]
    if POL_FOR_REF is not None and "polarity" in sub.columns:
        win = win[win["polarity"] == POL_FOR_REF]

    if win.empty:
        ref_auc[fname] = 0.0
        continue

    trace = (
        win.groupby("rt", as_index=False)["i"]
           .sum()
           .sort_values("rt")
    )

    ref_auc[fname] = 0.0 if len(trace) < 2 else float(np.trapz(trace["i"], trace["rt"]))

missing_refs = [k for k, v in ref_auc.items() if v == 0.0]
print("\n[INFO] Files missing reference compound (MS1 AUC==0):")
for f in missing_refs:
    print(" -", f)

ref_auc_df = pd.DataFrame([{"source_file": k, "ref_ms1_auc": v} for k, v in ref_auc.items()])
ref_auc_csv = f"reference_ms1_auc_AR_{ts}.csv"
ref_auc_df.to_csv(ref_auc_csv, index=False)
print("[DONE] wrote:", ref_auc_csv)

# ------------------------------------------------------------
# 6) Compute MS1 AUC for each (feature,file) from MS1 points
# ------------------------------------------------------------
perfile_summary = explode_matches_to_per_file(indiv_summary)

perfile_summary_auc = add_ms1_auc_from_points(
    perfile_summary,
    ms1_points,
    tol_mz=TOL,          # same tolerance you use elsewhere
    rt_pad=0.25,
    polarity=POLARITY if isinstance(POLARITY, int) else None,  # 1/-1/None
    intensity_col="i",   # integrate raw intensity
    mz_col="precmz"
)

# Add reference + normalized column
perfile_summary_auc = perfile_summary_auc.merge(ref_auc_df, on="source_file", how="left")
perfile_summary_auc["ms1_auc_over_ref"] = np.where(
    (perfile_summary_auc["ref_ms1_auc"].notna()) & (perfile_summary_auc["ref_ms1_auc"] > 0),
    perfile_summary_auc["ms1_auc"] / perfile_summary_auc["ref_ms1_auc"],
    np.nan
)

# Optional: pool back to one row per merged precursor (sum across files)
pooled_auc = pool_auc_back_to_matches(perfile_summary_auc)

# ------------------------------------------------------------
# 7) Save AUC outputs
# ------------------------------------------------------------
out_dir = Path("AUC_outputs")
out_dir.mkdir(exist_ok=True)

perfile_out = out_dir / f"MC_indiv_summary_perfile_with_ms1_auc_{ts}.csv"
pooled_out  = out_dir / f"MC_indiv_summary_pooled_ms1_auc_{ts}.csv"

perfile_summary_auc.to_csv(perfile_out, index=False)
pooled_auc.to_csv(pooled_out, index=False)

print("[DONE] Saved per-file AUC table:", perfile_out)
print("[DONE] Saved pooled AUC table:  ", pooled_out)

# ------------------------------------------------------------
# 8) Downstream scatter plot — use indiv_summary
# ------------------------------------------------------------
from indiv_combo_dot_plot import plot_indiv_scatter
fig, ax, path = plot_indiv_scatter(indiv_summary, out_dir="DotPlot_individual_ion_plots", fmt="png")

# ------------------------------------------------------------
# 9) (Optional) Your adduct pipeline block (unchanged)
# ------------------------------------------------------------
import adduct_pipeline as ap
import adduct_finder as af

def make_summary_ind_with_labels(df, *args, **kwargs):
    return make_summary_ind(df, *args, ion_to_label=ION_TO_LABEL, **kwargs)

filtered_ind = ind_hits_l[(ind_hits_l["precmz"] >= 650) & (ind_hits_l["precmz"] <= 1200)]

merged_summary, merged_edges, G = ap.run_merged(
    filtered_ind,
    make_summary_ind=make_summary_ind_with_labels,
    af_module=af,
    mz_col="merged_precmz",
    charge_col="charge",
    rt_col="rt_median",
    out_dir="Adduct_and_summary_outputs",
    save_graph=True,
)

# ------------------------------------------------------------
# 10) Library matching + tiles (unchanged)
# ------------------------------------------------------------
from cyanometdb_match import (
    load_library, read_any_table, match_ms1_to_lib, select_ms1_columns,
    _latest_indiv_summary, write_outputs, plot_matched_tiles
)

LIB_XLSX = "/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project/CyanoMetDB_Version03.xlsx"

lib_df  = load_library(LIB_XLSX, class_filter="Microcystin", sheet_index=1)
ms1_csv_for_match = _latest_indiv_summary()
ms1_df_for_match  = read_any_table(ms1_csv_for_match)

ms1_sel = select_ms1_columns(ms1_df_for_match)

matches = match_ms1_to_lib(ms1_sel, lib_df, tol_da=0.1)
matched_only = matches[matches["Compound identifier"].notna()].copy()

ts2 = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir_ts, paths = write_outputs(matches, matched_only, out_dir="CyanoMetDB_matches_out", ts=ts2)
plot_matched_tiles(matched_only, out_dir_ts=out_dir_ts, ts=ts2)

# ------------------------------------------------------------
# 11) Your remaining utilities (unchanged)
# ------------------------------------------------------------
from sum_intensity_from_scans import sum_intensities
df, outfile = sum_intensities()

from ms2_tilemap_intensities import plot_has_tilemap_from_latest
df, run_dir, summary_file = plot_has_tilemap_from_latest(ion_to_label=ION_TO_LABEL)

from group_run_files import group_files_by_timestamp
group_files_by_timestamp(custom_name="MC", minutes=7)


micropeptin:

In [ ]:
# ============================================================
# MP pipeline (MassQL diagnostic ions) + MS1 AUC from MS1 points
#updated for 1/5/2025 (12pm)
# ============================================================
# Assumes you already defined:
#   FILES, mu, MPIONS, TOL, POLARITY, RT_WINDOW, ION_TO_LABEL
#
# Also assumes you've already generated MS1 point tables earlier:
#   ms1_points_*.csv with columns:
#   source_file, scan, rt, precmz, i, mslevel, polarity
#
# Outputs:
#   individual_hits_MP_<ts>.csv
#   reference_ms1_auc_MP_<ts>.csv
#   AUC_outputs/MP_indiv_summary_perfile_with_ms1_auc_<ts>.csv
#   AUC_outputs/MP_indiv_summary_pooled_ms1_auc_<ts>.csv
#   (plus your plots)

import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from summary_builder import (
    make_summary_ind,
    explode_matches_to_per_file,
    add_ms1_auc_from_points,
    pool_auc_back_to_matches,
)

# ------------------------------------------------------------
# 0) Timestamp for all outputs in this run
# ------------------------------------------------------------
ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# ------------------------------------------------------------
# 1) MassQL query + run across files (MP diagnostic ions)
# ------------------------------------------------------------
query = mu.build_massql_query(
    ions_mz=ARIONS,
    tol_mz=TOL,
    polarity=POLARITY,
    rt_window=None
)

ind_hits = mu.run_across_files_individual(
    FILES, MPIONS, tol_mz=TOL, polarity=POLARITY, rt_window=RT_WINDOW
)

ind_hits_l = mu.add_MP_labels(ind_hits, ION_TO_LABEL, label_col="CyanopeptideClass_MP")

ind_csv = f"individual_hits_MP_{ts}.csv"
ind_hits_l.to_csv(ind_csv, index=False)
print("[DONE] wrote:", ind_csv)

# ------------------------------------------------------------
# 2) Plots
# ------------------------------------------------------------
from rt_histograms import plot_rt_histograms
from rt_histograms import load_latest_hits

# Reload latest (your helper likely picks up the CSV you just wrote)
ind_hits_l = load_latest_hits()

plot_rt_histograms(ind_hits_l, ION_TO_LABEL, out_dir_root="RT_histogram_plots")

from cyanopeptide_counts_plots import plot_indiv_counts
plot_indiv_counts(ind_hits_l, out_dir="plots_diagnostic_ion_counts")

from rt_mz_plot import plot_precursor_rt, plot_per_file_legend
fig, ax = plot_precursor_rt(ind_hits_l, ion_to_label=ION_TO_LABEL, save=True)
plot_per_file_legend(ind_hits_l, save=True)

from plotting_ind_heatmap import plot_heatmaps
plot_heatmaps(
    ind_hits_l,
    ion_to_label=ION_TO_LABEL,
    save=True,
    out_dir="Heatmap_Individual_ion_plots",
    fmt="png"
)

from filter_precursor_range_script import filter_precursor_range

ind_hits_filtered_range = filter_precursor_range(ind_hits_l, min_mz=650, max_mz=1400)
plot_heatmaps(
    ind_hits_filtered_range,
    ion_to_label=ION_TO_LABEL,
    BIN_WIDTH=0.1,
    merged=True,
    per_file=True
)

# ------------------------------------------------------------
# 3) Build merged precursor summary from MS2 hits
# ------------------------------------------------------------
indiv_summary = make_summary_ind(ind_hits_l, ion_to_label=ION_TO_LABEL)

# ------------------------------------------------------------
# 4) Load the most recent MS1 points CSV automatically
# ------------------------------------------------------------
ms1_csv = max(Path(".").glob("ms1_points_*.csv"), key=lambda p: p.stat().st_mtime)
print("[INFO] Using MS1 points file:", ms1_csv)

ms1_points = pd.read_csv(ms1_csv)

# normalize filenames to basenames (critical to match summary_builder logic)
ms1_points["source_file"] = ms1_points["source_file"].astype(str).apply(lambda x: Path(x).name)

# sanity checks
print("[INFO] MS1 points rows:", len(ms1_points))
print("[INFO] MS1 points unique files:", ms1_points["source_file"].nunique())
if "mslevel" in ms1_points.columns:
    print("[INFO] MS levels:", ms1_points["mslevel"].value_counts().head())

# ------------------------------------------------------------
# 5) Reference compound MS1 AUC per file (FROM MS1 POINTS)
# ------------------------------------------------------------
#edit this for reference compounds!!

REF_MZ = 192.20
REF_TOL = 0.1
REF_RT_MIN, REF_RT_MAX = RT_WINDOW  # minutes

# Optional: polarity filter on reference extraction
# (Only if your MS1 table has polarity populated reliably)
POL_FOR_REF = POLARITY if isinstance(POLARITY, int) else None

ref_auc = {}
for fname, sub in ms1_points.groupby("source_file", sort=False):
    win = sub[
        (sub["mslevel"] == 1)
        & (sub["precmz"].between(REF_MZ - REF_TOL, REF_MZ + REF_TOL))
        & (sub["rt"].between(REF_RT_MIN, REF_RT_MAX))
    ]
    if POL_FOR_REF is not None and "polarity" in sub.columns:
        win = win[win["polarity"] == POL_FOR_REF]

    if win.empty:
        ref_auc[fname] = 0.0
        continue

    trace = (
        win.groupby("rt", as_index=False)["i"]
           .sum()
           .sort_values("rt")
    )

    ref_auc[fname] = 0.0 if len(trace) < 2 else float(np.trapz(trace["i"], trace["rt"]))

missing_refs = [k for k, v in ref_auc.items() if v == 0.0]
print("\n[INFO] Files missing reference compound (MS1 AUC==0):")
for f in missing_refs:
    print(" -", f)

ref_auc_df = pd.DataFrame([{"source_file": k, "ref_ms1_auc": v} for k, v in ref_auc.items()])
ref_auc_csv = f"reference_ms1_auc_MP_{ts}.csv"
ref_auc_df.to_csv(ref_auc_csv, index=False)
print("[DONE] wrote:", ref_auc_csv)

# ------------------------------------------------------------
# 6) Compute MS1 AUC for each (feature,file) from MS1 points
# ------------------------------------------------------------
perfile_summary = explode_matches_to_per_file(indiv_summary)

perfile_summary_auc = add_ms1_auc_from_points(
    perfile_summary,
    ms1_points,
    tol_mz=TOL,          # same tolerance you use elsewhere
    rt_pad=0.25,
    polarity=POLARITY if isinstance(POLARITY, int) else None,  # 1/-1/None
    intensity_col="i",   # integrate raw intensity
    mz_col="precmz"
)

# Add reference + normalized column
perfile_summary_auc = perfile_summary_auc.merge(ref_auc_df, on="source_file", how="left")
perfile_summary_auc["ms1_auc_over_ref"] = np.where(
    (perfile_summary_auc["ref_ms1_auc"].notna()) & (perfile_summary_auc["ref_ms1_auc"] > 0),
    perfile_summary_auc["ms1_auc"] / perfile_summary_auc["ref_ms1_auc"],
    np.nan
)

# Optional: pool back to one row per merged precursor (sum across files)
pooled_auc = pool_auc_back_to_matches(perfile_summary_auc)

# ------------------------------------------------------------
# 7) Save AUC outputs
# ------------------------------------------------------------
out_dir = Path("AUC_outputs")
out_dir.mkdir(exist_ok=True)

perfile_out = out_dir / f"MP_indiv_summary_perfile_with_ms1_auc_{ts}.csv"
pooled_out  = out_dir / f"MP_indiv_summary_pooled_ms1_auc_{ts}.csv"

perfile_summary_auc.to_csv(perfile_out, index=False)
pooled_auc.to_csv(pooled_out, index=False)

print("[DONE] Saved per-file AUC table:", perfile_out)
print("[DONE] Saved pooled AUC table:  ", pooled_out)

# ------------------------------------------------------------
# 8) Downstream scatter plot — use indiv_summary
# ------------------------------------------------------------
from indiv_combo_dot_plot import plot_indiv_scatter
fig, ax, path = plot_indiv_scatter(indiv_summary, out_dir="DotPlot_individual_ion_plots", fmt="png")

# ------------------------------------------------------------
# 9) (Optional) Your adduct pipeline block (unchanged)
# ------------------------------------------------------------
import adduct_pipeline as ap
import adduct_finder as af

def make_summary_ind_with_labels(df, *args, **kwargs):
    return make_summary_ind(df, *args, ion_to_label=ION_TO_LABEL, **kwargs)

filtered_ind = ind_hits_l[(ind_hits_l["precmz"] >= 650) & (ind_hits_l["precmz"] <= 1200)]

merged_summary, merged_edges, G = ap.run_merged(
    filtered_ind,
    make_summary_ind=make_summary_ind_with_labels,
    af_module=af,
    mz_col="merged_precmz",
    charge_col="charge",
    rt_col="rt_median",
    out_dir="Adduct_and_summary_outputs",
    save_graph=True,
)

# ------------------------------------------------------------
# 10) Library matching + tiles (unchanged)
# ------------------------------------------------------------
from cyanometdb_match import (
    load_library, read_any_table, match_ms1_to_lib, select_ms1_columns,
    _latest_indiv_summary, write_outputs, plot_matched_tiles
)

LIB_XLSX = "/nfs/turbo/lsa-gdick2/geomicro/home/sheffera/ms2_project/CyanoMetDB_Version03.xlsx"

lib_df  = load_library(LIB_XLSX, class_filter="Cyanopeptolin", sheet_index=1)
ms1_csv_for_match = _latest_indiv_summary()
ms1_df_for_match  = read_any_table(ms1_csv_for_match)

ms1_sel = select_ms1_columns(ms1_df_for_match)

matches = match_ms1_to_lib(ms1_sel, lib_df, tol_da=0.1)
matched_only = matches[matches["Compound identifier"].notna()].copy()

ts2 = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir_ts, paths = write_outputs(matches, matched_only, out_dir="CyanoMetDB_matches_out", ts=ts2)
plot_matched_tiles(matched_only, out_dir_ts=out_dir_ts, ts=ts2)

# ------------------------------------------------------------
# 11) Your remaining utilities (unchanged)
# ------------------------------------------------------------
from sum_intensity_from_scans import sum_intensities
df, outfile = sum_intensities()

from ms2_tilemap_intensities import plot_has_tilemap_from_latest
df, run_dir, summary_file = plot_has_tilemap_from_latest(ion_to_label=ION_TO_LABEL)

from group_run_files import group_files_by_timestamp
group_files_by_timestamp(custom_name="MP", minutes=7)
